# Is Your Score Real? — The Contamination Test

You just ran an investigation agent against BOTSv3 and got a score.

**Here is the uncomfortable question: how much of that score came from investigating, and how much came from the model already knowing the answers?**

BOTSv3 is a public dataset. The questions, the answers, and hundreds of walkthrough blog posts are all on the open internet — which means they are very likely in the training data of every frontier model.

The test is simple: **ask the same questions with no tools at all.**

- If the model answers correctly with **no data access**, it memorized the answer.
- If it says *"I don't know"*, it is being honest about its limits.
- If it only answers correctly **with** Splunk, it actually investigated.

> Failure is the desired outcome here. A model that scores 0% on this notebook is a model whose benchmark score you can trust.

## The test data

Same six B2 questions from the eval lab — loaded from the canonical file so there is one source of truth.

In [ ]:
import json
import os
import re
import subprocess
import time
from pathlib import Path

# Canonical question set — same file the eval lab uses.
TESTS_PATH = Path(os.environ.get("B2_TESTS_PATH", "../lab2/botsv3_b2_tests.json"))

with TESTS_PATH.open() as f:
    B2_TESTS = json.load(f)

print(f"{len(B2_TESTS)} questions loaded from {TESTS_PATH}")
for qid, q in B2_TESTS.items():
    print(f"  [{q['difficulty']:>6}] {q['name']}")

## Answer checking

Two pieces, both borrowed from how `bots-bench` grades runs:

1. **`extract_answer()`** — pull the payload out of an `ANSWER: <value>` line, tolerating markdown bold and list bullets.
2. **`check_answer()`** — the same substring + `re:`-prefix regex matcher used in the eval lab.

In [ ]:
# Mirrors bots-bench ANSWER_LINE_RE / extract_answer_payload()
ANSWER_LINE_RE = re.compile(r"^(?:[-*]\s*)?ANSWER\s*:\s*(.*)\s*$", flags=re.IGNORECASE)


def extract_answer(text: str) -> str:
    """Pull the payload from an 'ANSWER: <value>' line, else return the whole text."""
    for line in (text or "").splitlines():
        for candidate in (line.strip(), line.strip().replace("**", "").replace("__", "")):
            m = ANSWER_LINE_RE.match(candidate)
            if m:
                return m.group(1).strip().strip("`").strip()
    return (text or "").strip()


def check_answer(output: str, patterns: list[str]) -> bool:
    """Same matcher as the eval lab: substring, or regex when prefixed with 're:'."""
    for pattern in patterns:
        if pattern.startswith("re:"):
            if re.search(pattern[3:], output, re.IGNORECASE):
                return True
        elif pattern.lower() in output.lower():
            return True
    return False


def said_unknown(text: str) -> bool:
    """Did the model honestly decline instead of guessing?"""
    return bool(re.search(r"\b(i don'?t know|unknown|not enough information|cannot determine)\b",
                          text or "", re.IGNORECASE))


# Sanity checks
print(extract_answer("Some reasoning...\n**ANSWER: 5244329601**"))          # 5244329601
print(check_answer("Xenial Xerus", ["re:(?i)xenial\\s+xerus"]))              # True
print(said_unknown("ANSWER: I don't know"))                                  # True

## Why stripping the tool list matters

It is not enough to just "not connect Splunk." If the Splunk MCP tools are still *declared*, their names sit in the context window and prime the model toward the right domain — which quietly inflates the closed-book score.

`bots-bench` handles this at the CLI flag layer, with the comment:

> *"Strip MCP tools from no-tools runs to prevent contextual priming. Without this, the tool list leaks domain info (e.g., Splunk MCP tool names)."*

So a genuine closed-book run needs **all three** flags:

| Flag | Why |
| :--- | :--- |
| `--tools ""` | No built-in tools (no Bash, Read, WebSearch) |
| `--mcp-config '{"mcpServers":{}}'` | No MCP servers |
| `--strict-mcp-config` | Ignore any MCP config inherited from the environment |

Plus `--no-session-persistence`, so run N cannot read the answer out of run N-1's memory.

In [ ]:
MODEL = os.environ.get("BENCH_MODEL", "claude-opus-4-7")
TIMEOUT_SEC = int(os.environ.get("BENCH_TIMEOUT", "120"))

# Verbatim from bots-bench no_tools_prompt(cross_validate=False)
NO_TOOLS_PROMPT = (
    "Do NOT use any tools. Answer directly from your own knowledge. "
    "If you are unsure, say 'I don't know.' "
    "Return only one line: 'ANSWER: <answer>'. "
    "If answer has multiple values, use comma-separated values with no spaces and no prose. "
    "Do not transform raw values from evidence.\nQuestion: {question}"
)


def run_closed_book(question: str, model: str = MODEL, timeout: int = TIMEOUT_SEC) -> dict:
    """Ask one question with every tool surface removed."""
    cmd = [
        "claude", "-p", NO_TOOLS_PROMPT.format(question=question),
        "--model", model,
        "--tools", "",
        "--mcp-config", '{"mcpServers":{}}',
        "--strict-mcp-config",
        "--no-session-persistence",
    ]
    start = time.time()
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        raw = proc.stdout
    except subprocess.TimeoutExpired:
        raw = ""
    return {
        "raw": raw,
        "answer": extract_answer(raw),
        "elapsed": time.time() - start,
    }


print(f"Model under test: {MODEL}")
print("Closed-book prompt:\n")
print(NO_TOOLS_PROMPT.format(question="<question goes here>"))

## Run the closed-book test

Six questions, one run each. This is fast and cheap — no Splunk, no tools, one short answer per question.

In [ ]:
NUM_RUNS = 1  # Raise to 3 if you want a rate rather than a single sample

closed_book = []

for qid, q in B2_TESTS.items():
    print(f"\n--- {q['name']} [{q['difficulty']}] ---")
    passes = 0
    unknowns = 0
    last_answer = ""

    for run in range(NUM_RUNS):
        result = run_closed_book(q["question"])
        answer = result["answer"]
        last_answer = answer

        correct = check_answer(answer, q["expected_patterns"])
        unknown = said_unknown(answer)
        passes += correct
        unknowns += unknown

        label = "RECALLED" if correct else ("unknown" if unknown else "wrong")
        print(f"  Run {run}: {label:>9} ({result['elapsed']:.1f}s) — {answer[:70]}")

    closed_book.append({
        "id": qid,
        "name": q["name"],
        "difficulty": q["difficulty"],
        "expected": q["expected_output"],
        "closed_book_passes": passes,
        "unknowns": unknowns,
        "runs": NUM_RUNS,
        "closed_book_rate": passes / NUM_RUNS,
        "last_answer": last_answer,
    })

recalled = sum(r["closed_book_passes"] > 0 for r in closed_book)
print(f"\n{recalled}/{len(closed_book)} questions answered with NO data access.")

## The comparison table

Now put the closed-book result next to your **with-tools** score from the eval lab.

Fill in `WITH_TOOLS_RATES` from your own run — that is the whole point. Your numbers, your model, your verdict.

In [ ]:
import pandas as pd

# EDIT ME: paste your with-tools pass rates from the eval lab (0.0 - 1.0).
# Leave as None for any question you did not run.
WITH_TOOLS_RATES = {
    "b2_q19_support_case_id": None,
    "b2_q22_user_agent": None,
    "b2_q18_iam_key_errors": None,
    "b2_q20_secret_access_key": None,
    "b2_q21_target_resource": None,
    "b2_q23_ubuntu_codename": None,
}


def verdict(closed_rate: float, tools_rate) -> str:
    if closed_rate > 0 and (tools_rate is None or tools_rate <= closed_rate):
        return "MEMORIZED — score is not measuring investigation"
    if closed_rate > 0:
        return "PARTIAL — some recall, some investigation"
    if tools_rate is None:
        return "clean closed-book (run with tools to compare)"
    if tools_rate > 0:
        return "INVESTIGATED — earned with evidence"
    return "unsolved in both modes"


rows = []
for r in closed_book:
    tools_rate = WITH_TOOLS_RATES.get(r["id"])
    rows.append({
        "Question": r["name"][:44],
        "Difficulty": r["difficulty"],
        "Closed-book": f"{r['closed_book_rate']:.0%}",
        "With Splunk": "—" if tools_rate is None else f"{tools_rate:.0%}",
        "Verdict": verdict(r["closed_book_rate"], tools_rate),
    })

df = pd.DataFrame(rows)
print(f"Model: {MODEL}   |   runs per question: {NUM_RUNS}\n")
display(df)

## How to read this

| Pattern | What it means | What to do |
| :--- | :--- | :--- |
| Correct closed-book | **Memorized.** The question tests recall, not investigation. | Drop it from your benchmark, or treat the score as a ceiling, not a measurement. |
| *"I don't know"* closed-book, correct with tools | **Investigated.** This is the honest case. | Keep it. This question is doing real work. |
| Confidently wrong closed-book | **Confabulation.** The model guesses rather than declining. | Worrying in an agent loop — a wrong answer with no hedge propagates downstream. |
| Wrong in both modes | Genuinely hard, or needs knowledge outside the data. | Fine to keep, but it is measuring something other than SPL skill. |

**Watch Q6 (Ubuntu codename) especially.** In the eval lab it scores near 0% *with* Splunk, because the AMI-ID-to-codename mapping is not in the dataset. If it comes back correct closed-book, you have a clean inversion: the model knows the answer but cannot derive it. That single cell makes the whole argument.

## Optional: the paraphrase probe

A stronger test than the binary one. Keep the *facts* identical but rewrite the wording so it looks nothing like the published BOTS phrasing.

If accuracy collapses under paraphrase, the model was pattern-matching the **question text**, not recalling the **fact**.

In [ ]:
PARAPHRASED = {
    "b2_q22_user_agent": (
        "An AWS API call to describe account details was made with credentials that had "
        "been published to a public repo. Which client software string appeared in that request?"
    ),
    "b2_q23_ubuntu_codename": (
        "Someone tried to spin up a Canonical Linux machine image using stolen cloud "
        "credentials. What was the two-word release nickname of that OS version?"
    ),
}

for qid, paraphrase in PARAPHRASED.items():
    q = B2_TESTS[qid]
    original = run_closed_book(q["question"])["answer"]
    rewritten = run_closed_book(paraphrase)["answer"]

    print(f"\n--- {q['name']} ---")
    print(f"  original phrasing : {'PASS' if check_answer(original, q['expected_patterns']) else 'FAIL'} — {original[:60]}")
    print(f"  paraphrased       : {'PASS' if check_answer(rewritten, q['expected_patterns']) else 'FAIL'} — {rewritten[:60]}")

## What real rigor looks like

This notebook is the teaching version. A production benchmark has to close several more holes, because an agent that *wants* to score well has more than one way to cheat:

- **Tool-surface control** — `read` / `grep` / `find` / `ls` are never exposed, so the agent cannot go read the answer key off disk.
- **Network isolation** — run the agent on an internal subnet whose only egress is an allowlist proxy, so `bash` + `curl` cannot fetch a walkthrough blog mid-run.
- **Trace auditing** — classify every run by the contamination vectors it actually touched (web search, shell commands, host-path reads) and mark it green / yellow / red.
- **Cross-provider judging** — never let a provider grade its own model. A Claude run gets an OpenAI judge, and vice versa.
- **Held-out datasets** — the real answer to contamination is a corpus the models have never seen. Public CTFs can only ever be a cross-check.

`bots-bench` (the benchmark behind [botsbench.com](https://botsbench.com)) implements all of these. It publishes both a private held-out CTF as the primary score *and* BOTSv3 as an explicit contamination cross-check — which is exactly the pairing this notebook demonstrates in miniature.

### The takeaway

Any benchmark number without a contamination control is unfalsifiable. Before you trust a vendor's accuracy claim — or your own — ask what the score is when you take the data away.